# EfficientNet 안전고리 분류 v2

- 설명: RF-DETR 검출 결과와 개선된 EfficientNet 분류기를 결합한 후속 실험입니다.
- 작성자: 조하민

> 공개 저장소에는 데이터, 모델 가중치, API 키와 노트북 실행 결과를 포함하지 않습니다.


In [ ]:
# 1
!pip install -q "rfdetr[train,loggers]" supervision opencv-python pillow

In [ ]:
# 3
from pathlib import Path
import tempfile
import inspect

import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

from rfdetr import RFDETRSegLarge


# ============================================================
# 경로 설정
# ============================================================

# RF-DETR 체크포인트
RFDETR_CHECKPOINT = "/content/drive/MyDrive/checkpoint_best_ema.pth"

# EfficientNet 체크포인트
EFFNET_CHECKPOINT = "/content/drive/MyDrive/best_effnet_hook_classifier.pth"

# 입력/출력 영상
# SOURCE_VIDEO는 원본 영상 경로입니다. 영상 자르기 셀을 여러 번 실행해도 바꾸지 마세요.
SOURCE_VIDEO = "/content/drive/MyDrive/input_video.mp4"
INPUT_VIDEO = SOURCE_VIDEO
OUTPUT_VIDEO = "/content/output_hook_status.mp4"


# ============================================================
# 모델 설정
# ============================================================

# RF-DETR에서 hook에 해당하는 class_id.
# 처음에는 0으로 두고, 결과가 안 나오면 아래 "class_id 확인 셀"로 확인하세요.
HOOK_CLASS_ID = 2

# RF-DETR class_id -> class name.
# 본인 데이터셋 순서에 맞게 수정하세요.
# 예: 0=hook, 1=lifeline, 2=worker, 3=harness 라면 아래처럼 둡니다.
DETECTOR_CLASS_NAMES = {
    1: "harness",
    2: "hook",
    3: "lanyard",
    4: "lifeline",
    5: "worker",


}

# RF-DETR 학습 클래스 개수.
# class count mismatch 에러가 나면 예: hook/lifeline/worker/harness = 4로 설정.
RFDETR_NUM_CLASSES = None

# EfficientNet 분류 클래스 순서.
# efficient.ipynb의 ImageFolder 기준: connected -> 0, unconnected -> 1
CLASS_NAMES = ["connected", "unconnected"]


# ============================================================
# threshold / crop 설정
# ============================================================

# RF-DETR 탐지 threshold.
# 낮추면 hook 후보가 더 많이 나오지만 오탐도 늘어납니다.
# 아무것도 탐지되지 않으면 0.50 -> 0.25 -> 0.10 순서로 낮춰보세요.
RFDETR_THRESHOLD = 0.70

# EfficientNet unconnected 분류 threshold.
# unconnected_prob가 이 값 이상이면 UNCONNECTED로 빨간 박스를 표시합니다.
# 안전 쪽에서는 놓치는 것보다 경고가 많은 편이 낫기 때문에
# 0.30~0.50 사이를 실험해보는 것을 권장합니다.
UNCONNECTED_THRESHOLD = 0.20

# crop 여백입니다. hook 박스가 너무 타이트하면 0.50 -> 0.80으로 올려보세요.
CROP_MARGIN_RATIO = 0.50

# 9번 class_id 확인 셀에서 쓰는 테스트 threshold는
# 9번 셀 안의 TEST_THRESHOLD 값을 따로 바꾸면 됩니다.

# 디버깅용 hook crop 저장 여부
SAVE_DEBUG_CROPS = True
DEBUG_CROP_DIR = "/content/hook_debug_crops"

# 영상 처리 속도 조절
# 1이면 모든 프레임 처리, 2이면 2프레임마다 1번 처리
FRAME_STRIDE = 1
WRITE_SKIPPED_FRAMES = True

# 모든 RF-DETR 클래스를 표시할지 여부.
# True면 hook 외 lifeline/worker/harness 등도 박스로 표시합니다.
DRAW_ALL_CLASSES = True

# 박스 깜박임 완화 설정.
# 탐지가 1~몇 프레임 빠져도 이전 박스를 잠깐 유지합니다.
ENABLE_TEMPORAL_SMOOTHING = True
MAX_MISSING_FRAMES = 5
TRACK_IOU_THRESHOLD = 0.30
BOX_SMOOTHING_ALPHA = 0.65

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
# 4
def trim_video_opencv(input_path, output_path, start_sec, end_sec=None, duration_sec=None):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open input video: {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        fps = 30

    video_duration = total_frames / fps if total_frames > 0 else 0
    start_sec = max(0, float(start_sec))

    if video_duration > 0 and start_sec >= video_duration:
        cap.release()
        raise ValueError(
            f"start_sec is outside the video duration: "
            f"start_sec={start_sec}, duration={round(video_duration, 2)}. "
            "Check SOURCE_VIDEO and START_SEC."
        )

    if duration_sec is not None:
        end_sec = start_sec + float(duration_sec)
    elif end_sec is not None:
        end_sec = float(end_sec)
    else:
        cap.release()
        raise ValueError("Set either end_sec or duration_sec.")

    if video_duration > 0:
        end_sec = min(end_sec, video_duration)

    if end_sec <= start_sec:
        cap.release()
        raise ValueError(
            f"end_sec must be greater than start_sec: {start_sec} -> {end_sec}. "
            "If you want to cut N seconds from START_SEC, use DURATION_SEC=N."
        )

    start_frame = int(start_sec * fps)
    end_frame = int(end_sec * fps)

    output_path = str(output_path)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    current_frame = start_frame
    written = 0

    while current_frame < end_frame:
        ok, frame = cap.read()
        if not ok:
            break

        writer.write(frame)
        written += 1
        current_frame += 1

    cap.release()
    writer.release()

    print("Original video:", input_path)
    print("Trimmed video:", output_path)
    print("FPS:", fps)
    print("Original duration(sec):", round(video_duration, 2))
    print("Trim range(sec):", start_sec, "->", end_sec)
    print("Written frames:", written)

    return output_path


# ============================================================
# 여기만 수정하세요
# ============================================================

START_SEC = 720

# 둘 중 하나만 사용하세요.
# 방법 1: 시작 초와 끝 초를 직접 지정
END_SEC = 750

# 방법 2: 시작 초부터 몇 초 동안 자를지 지정
# 예: START_SEC = 800, DURATION_SEC = 10 이면 800초부터 810초까지 자름
DURATION_SEC = None

TRIMMED_VIDEO = "/content/input_trimmed.mp4"

INPUT_VIDEO = trim_video_opencv(
    input_path=SOURCE_VIDEO,
    output_path=TRIMMED_VIDEO,
    start_sec=START_SEC,
    end_sec=END_SEC,
    duration_sec=DURATION_SEC,
)

print("Now INPUT_VIDEO is:", INPUT_VIDEO)

In [ ]:
# 5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

effnet_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def build_effnet(num_classes=2):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def load_effnet_classifier(checkpoint_path):
    model = build_effnet(num_classes=len(CLASS_NAMES)).to(DEVICE)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict)
    model.eval()
    return model


@torch.no_grad()
def classify_hook_crop(crop_rgb, classifier, threshold=UNCONNECTED_THRESHOLD):
    image = Image.fromarray(crop_rgb).convert("RGB")
    x = effnet_transform(image).unsqueeze(0).to(DEVICE)

    logits = classifier(x)
    probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()

    connected_idx = CLASS_NAMES.index("connected")
    unconnected_idx = CLASS_NAMES.index("unconnected")

    connected_prob = float(probs[connected_idx])
    unconnected_prob = float(probs[unconnected_idx])

    if unconnected_prob >= threshold:
        pred_label = "unconnected"
    else:
        pred_label = "connected"

    return {
        "label": pred_label,
        "connected_prob": connected_prob,
        "unconnected_prob": unconnected_prob,
    }

In [ ]:
# 6
def print_checkpoint_summary(checkpoint_path, title="checkpoint"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    print(f"{title} type:", type(checkpoint))
    if isinstance(checkpoint, dict):
        keys = list(checkpoint.keys())
        print(f"{title} top-level keys:", keys[:30])
    return checkpoint


def _looks_like_state_dict(value):
    return (
        isinstance(value, dict)
        and len(value) > 0
        and any(isinstance(v, torch.Tensor) for v in value.values())
    )


def _strip_prefix_from_state_dict(state_dict, prefix):
    if not all(isinstance(k, str) for k in state_dict.keys()):
        return state_dict
    if not any(k.startswith(prefix) for k in state_dict.keys()):
        return state_dict
    return {
        k[len(prefix):] if k.startswith(prefix) else k: v
        for k, v in state_dict.items()
    }


def _find_rfdetr_state_dict(checkpoint):
    if not isinstance(checkpoint, dict):
        raise ValueError("RF-DETR checkpoint is not a dictionary.")

    if _looks_like_state_dict(checkpoint):
        return checkpoint

    candidate_keys = [
        "model",
        "ema",
        "model_ema",
        "state_dict",
        "model_state_dict",
        "module",
        "net",
    ]

    for key in candidate_keys:
        value = checkpoint.get(key)
        if _looks_like_state_dict(value):
            return value
        if isinstance(value, dict):
            for nested_key in candidate_keys:
                nested_value = value.get(nested_key)
                if _looks_like_state_dict(nested_value):
                    return nested_value

    for value in checkpoint.values():
        if _looks_like_state_dict(value):
            return value
        if isinstance(value, dict):
            for nested_value in value.values():
                if _looks_like_state_dict(nested_value):
                    return nested_value

    raise KeyError(
        "Could not find RF-DETR model weights in checkpoint. "
        f"Top-level keys: {list(checkpoint.keys())[:30]}"
    )


def normalize_rfdetr_checkpoint(checkpoint_path):
    checkpoint = print_checkpoint_summary(checkpoint_path, title="RF-DETR checkpoint")

    if isinstance(checkpoint, dict) and "model" in checkpoint:
        return checkpoint_path

    state_dict = _find_rfdetr_state_dict(checkpoint)
    state_dict = _strip_prefix_from_state_dict(state_dict, "module.")
    state_dict = _strip_prefix_from_state_dict(state_dict, "model.")

    normalized = {"model": state_dict}

    if isinstance(checkpoint, dict):
        for key in ["args", "config", "model_config", "train_config", "epoch"]:
            if key in checkpoint:
                normalized[key] = checkpoint[key]

    normalized_path = Path(tempfile.gettempdir()) / "rfdetr_normalized_checkpoint.pth"
    torch.save(normalized, normalized_path)
    print("Normalized RF-DETR checkpoint saved to:", normalized_path)
    return str(normalized_path)


def build_rfdetr_model(checkpoint_path):
    kwargs = {"pretrain_weights": checkpoint_path}

    if RFDETR_NUM_CLASSES is not None:
        signature = inspect.signature(RFDETRSegLarge)
        if "num_classes" in signature.parameters:
            kwargs["num_classes"] = RFDETR_NUM_CLASSES
        else:
            print(
                "RFDETR_NUM_CLASSES was set, but this RFDETRSegLarge "
                "constructor does not expose a num_classes argument."
            )

    return RFDETRSegLarge(**kwargs)


def load_rfdetr_detector(checkpoint_path):
    try:
        return build_rfdetr_model(checkpoint_path)
    except KeyError as exc:
        if str(exc).strip("'\"") != "model":
            raise
        print(
            "RF-DETR loader expected checkpoint['model'], but this file uses "
            "a different checkpoint layout. Trying automatic normalization..."
        )
        normalized_path = normalize_rfdetr_checkpoint(checkpoint_path)
        return build_rfdetr_model(normalized_path)

In [ ]:
# 7
def predict_rfdetr_on_frame(detector, frame_rgb, threshold=RFDETR_THRESHOLD):
    pil_image = Image.fromarray(frame_rgb).convert("RGB")

    try:
        return detector.predict(images=pil_image, threshold=threshold)
    except Exception:
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=True) as tmp:
            pil_image.save(tmp.name, quality=95)
            return detector.predict(images=tmp.name, threshold=threshold)


def crop_with_margin(frame_rgb, xyxy, margin_ratio=CROP_MARGIN_RATIO):
    h, w = frame_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in xyxy]

    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(0, min(x2, w))
    y2 = max(0, min(y2, h))

    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)
    mx = int(bw * margin_ratio)
    my = int(bh * margin_ratio)

    cx1 = max(0, x1 - mx)
    cy1 = max(0, y1 - my)
    cx2 = min(w, x2 + mx)
    cy2 = min(h, y2 + my)

    crop = frame_rgb[cy1:cy2, cx1:cx2]
    return crop, (cx1, cy1, cx2, cy2), (x1, y1, x2, y2)


def get_detector_class_name(class_id):
    if class_id in DETECTOR_CLASS_NAMES:
        return DETECTOR_CLASS_NAMES[class_id]
    return f"class_{class_id}"


def iter_detections(detections):
    if detections is None:
        return

    if not hasattr(detections, "xyxy"):
        return

    class_ids = getattr(detections, "class_id", None)
    confidences = getattr(detections, "confidence", None)

    for i, xyxy in enumerate(detections.xyxy):
        class_id = int(class_ids[i]) if class_ids is not None else None
        confidence = float(confidences[i]) if confidences is not None else 0.0

        yield {
            "xyxy": xyxy,
            "class_id": class_id,
            "class_name": get_detector_class_name(class_id),
            "det_conf": confidence,
        }


def iter_hook_detections(detections, hook_class_id=HOOK_CLASS_ID):
    for det in iter_detections(detections):
        if det["class_id"] == hook_class_id:
            yield det


def draw_label(frame_bgr, text, origin, color):
    x, y = origin
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    thickness = 2
    text_size, baseline = cv2.getTextSize(text, font, font_scale, thickness)
    tw, th = text_size

    y = max(th + 8, y)
    cv2.rectangle(
        frame_bgr,
        (x, y - th - baseline - 6),
        (x + tw + 8, y + baseline),
        color,
        -1,
    )
    cv2.putText(
        frame_bgr,
        text,
        (x + 4, y - 4),
        font,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA,
    )


def draw_hook_result(frame_bgr, box, result, det_conf):
    x1, y1, x2, y2 = box

    if result["label"] == "unconnected":
        color = (0, 0, 255)
        status_text = "UNCONNECTED"
    else:
        color = (0, 180, 0)
        status_text = "CONNECTED"

    label = (
        f'{status_text} '
        f'd:{det_conf:.2f} '
        f'u:{result["unconnected_prob"]:.2f}'
    )

    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color, 3)
    draw_label(frame_bgr, label, (x1, y1 - 8), color)


CLASS_COLORS = {
    "hook": (0, 180, 255),
    "lifeline": (0, 255, 255),
    "worker": (255, 0, 0),
    "harness": (255, 0, 255),
}


def color_for_class(class_name):
    if class_name in CLASS_COLORS:
        return CLASS_COLORS[class_name]

    # Stable fallback color from class name.
    seed = abs(hash(class_name)) % 255
    return (
        int((seed * 37) % 255),
        int((seed * 17) % 255),
        int((seed * 97) % 255),
    )


def draw_detection_result(frame_bgr, box, class_name, det_conf, missed=0):
    x1, y1, x2, y2 = [int(v) for v in box]
    color = color_for_class(class_name)
    label = f"{class_name} d:{det_conf:.2f}"
    if missed > 0:
        label += f" hold:{missed}"

    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color, 2)
    draw_label(frame_bgr, label, (x1, y1 - 8), color)


def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0

    return inter_area / union


def smooth_box(old_box, new_box, alpha=BOX_SMOOTHING_ALPHA):
    old_box = np.array(old_box, dtype=np.float32)
    new_box = np.array(new_box, dtype=np.float32)
    smoothed = alpha * old_box + (1 - alpha) * new_box
    return tuple(int(round(v)) for v in smoothed)


def update_temporal_tracks(tracks, items, next_track_id):
    if not ENABLE_TEMPORAL_SMOOTHING:
        new_tracks = []
        for item in items:
            track = dict(item)
            track["id"] = next_track_id
            track["missed"] = 0
            next_track_id += 1
            new_tracks.append(track)
        return new_tracks, next_track_id

    matched_track_ids = set()
    matched_item_ids = set()

    for item_idx, item in enumerate(items):
        best_track_idx = None
        best_iou = 0.0

        for track_idx, track in enumerate(tracks):
            if track_idx in matched_track_ids:
                continue
            if track["class_id"] != item["class_id"]:
                continue

            iou = box_iou(track["box"], item["box"])
            if iou > best_iou:
                best_iou = iou
                best_track_idx = track_idx

        if best_track_idx is not None and best_iou >= TRACK_IOU_THRESHOLD:
            track = tracks[best_track_idx]
            track["box"] = smooth_box(track["box"], item["box"])
            track["det_conf"] = item["det_conf"]
            track["class_name"] = item["class_name"]
            track["is_hook"] = item["is_hook"]
            track["result"] = item.get("result")
            track["missed"] = 0
            matched_track_ids.add(best_track_idx)
            matched_item_ids.add(item_idx)

    for track_idx, track in enumerate(tracks):
        if track_idx not in matched_track_ids:
            track["missed"] += 1

    tracks = [
        track
        for track in tracks
        if track["missed"] <= MAX_MISSING_FRAMES
    ]

    for item_idx, item in enumerate(items):
        if item_idx in matched_item_ids:
            continue

        track = dict(item)
        track["id"] = next_track_id
        track["missed"] = 0
        next_track_id += 1
        tracks.append(track)

    return tracks, next_track_id


def draw_track(frame_bgr, track):
    if track.get("is_hook") and track.get("result") is not None:
        draw_hook_result(
            frame_bgr,
            track["box"],
            track["result"],
            det_conf=track["det_conf"],
        )
    else:
        draw_detection_result(
            frame_bgr,
            track["box"],
            track["class_name"],
            det_conf=track["det_conf"],
            missed=track.get("missed", 0),
        )

In [ ]:
# 8
print("Loading RF-DETR detector...")
detector = load_rfdetr_detector(RFDETR_CHECKPOINT)

print("Loading EfficientNet classifier...")
classifier = load_effnet_classifier(EFFNET_CHECKPOINT)

print("Models loaded.")

In [ ]:
# 9
TEST_DIR = Path("/content/rfdetr_class_id_test_frames")
TEST_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open INPUT_VIDEO: {INPUT_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("total_frames:", total_frames)

# 영상 전체에서 10개 지점을 골라 테스트합니다.
if total_frames > 0:
    sample_indices = np.linspace(0, max(0, total_frames - 1), num=10, dtype=int)
else:
    sample_indices = np.arange(10)

TEST_THRESHOLD = 0.10

found_any = False

for frame_idx in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame_bgr = cap.read()
    if not ok:
        print("Could not read frame:", frame_idx)
        continue

    test_image = str(TEST_DIR / f"frame_{int(frame_idx):06d}.jpg")
    cv2.imwrite(test_image, frame_bgr)

    detections = detector.predict(images=test_image, threshold=TEST_THRESHOLD)

    xyxy = getattr(detections, "xyxy", None)
    class_id = getattr(detections, "class_id", None)
    confidence = getattr(detections, "confidence", None)

    num_det = len(xyxy) if xyxy is not None else 0
    print("=" * 70)
    print("frame:", int(frame_idx), "image:", test_image)
    print("num detections:", num_det)
    print("class_id:", class_id)
    print("confidence:", confidence)

    if num_det > 0:
        found_any = True

cap.release()

if not found_any:
    print("=" * 70)
    print("No detections found even at threshold", TEST_THRESHOLD)
    print("Check these:")
    print("1. Does the sampled video actually contain visible hooks?")
    print("2. Is RFDETR_CHECKPOINT the trained hook detector checkpoint?")
    print("3. Was RF-DETR trained with the same architecture, RFDETRSegLarge?")
    print("4. Try lowering RFDETR_THRESHOLD or testing a known hook image.")

In [ ]:
# 10
# ============================================================
# 1) 체크포인트 내부 구조 확인
# ============================================================

ckpt = torch.load(RFDETR_CHECKPOINT, map_location="cpu")

print("checkpoint type:", type(ckpt))
if isinstance(ckpt, dict):
    print("top-level keys:", list(ckpt.keys())[:50])

state_dict = None

if isinstance(ckpt, dict):
    if "model" in ckpt and isinstance(ckpt["model"], dict):
        state_dict = ckpt["model"]
    else:
        try:
            state_dict = _find_rfdetr_state_dict(ckpt)
        except Exception as e:
            print("Could not find state_dict:", repr(e))

if state_dict is not None:
    print("state_dict keys sample:")
    for k in list(state_dict.keys())[:20]:
        print(" ", k, tuple(state_dict[k].shape) if hasattr(state_dict[k], "shape") else type(state_dict[k]))

    class_bias_keys = [k for k in state_dict.keys() if "class_embed" in k and "bias" in k]
    print("class_embed bias keys:", class_bias_keys[:20])

    for k in class_bias_keys[:5]:
        print(k, "shape:", tuple(state_dict[k].shape))
        print("RF-DETR class count is usually shape[0] - 1 =", state_dict[k].shape[0] - 1)


# ============================================================
# 2) 영상에서 테스트 프레임 저장 및 화면 표시
# ============================================================

TEST_IMAGE = "/content/rfdetr_debug_frame.jpg"

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open INPUT_VIDEO: {INPUT_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
middle_frame = max(0, total_frames // 2)
cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)

ok, frame_bgr = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Could not read debug frame from video.")

cv2.imwrite(TEST_IMAGE, frame_bgr)
print("Saved debug frame:", TEST_IMAGE)
print("frame shape:", frame_bgr.shape)

from IPython.display import display
display(Image.open(TEST_IMAGE))


# ============================================================
# 3) threshold를 아주 낮춰서 탐지 테스트
# ============================================================

for th in [0.50, 0.25, 0.10, 0.05, 0.01, 0.001]:
    detections = detector.predict(images=TEST_IMAGE, threshold=th)
    xyxy = getattr(detections, "xyxy", None)
    class_id = getattr(detections, "class_id", None)
    confidence = getattr(detections, "confidence", None)
    num_det = len(xyxy) if xyxy is not None else 0

    print("=" * 70)
    print("threshold:", th)
    print("num detections:", num_det)
    print("class_id:", class_id)
    print("confidence:", confidence)

In [ ]:
# 11
def run_video_pipeline():
    debug_dir = Path(DEBUG_CROP_DIR)
    if SAVE_DEBUG_CROPS:
        debug_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(INPUT_VIDEO)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open input video: {INPUT_VIDEO}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        fps = 30

    output_path = Path(OUTPUT_VIDEO)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    print("Input:", INPUT_VIDEO)
    print("Output:", OUTPUT_VIDEO)
    print("Frames:", total_frames, "FPS:", fps, "Size:", (width, height))

    frame_idx = 0
    processed_frames = 0
    total_hooks = 0
    total_unconnected = 0
    tracks = []
    next_track_id = 1

    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        should_process = frame_idx % FRAME_STRIDE == 0

        if should_process:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            detections = predict_rfdetr_on_frame(detector, frame_rgb)

            hook_count_this_frame = 0
            display_items = []

            for det in iter_detections(detections):
                raw_box = tuple(int(round(v)) for v in det["xyxy"])
                is_hook = det["class_id"] == HOOK_CLASS_ID
                result = None

                if is_hook:
                    crop_rgb, crop_box, original_box = crop_with_margin(
                        frame_rgb,
                        det["xyxy"],
                        margin_ratio=CROP_MARGIN_RATIO,
                    )

                    if crop_rgb.size == 0:
                        continue

                    result = classify_hook_crop(crop_rgb, classifier)
                    hook_count_this_frame += 1
                    total_hooks += 1

                    if result["label"] == "unconnected":
                        total_unconnected += 1

                    if SAVE_DEBUG_CROPS:
                        crop_name = (
                            f"frame_{frame_idx:06d}_"
                            f"hook_{hook_count_this_frame:02d}_"
                            f"{result['label']}_"
                            f"u{result['unconnected_prob']:.2f}.jpg"
                        )
                        crop_bgr = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2BGR)
                        cv2.imwrite(str(debug_dir / crop_name), crop_bgr)

                if is_hook or DRAW_ALL_CLASSES:
                    display_items.append({
                        "box": raw_box,
                        "class_id": det["class_id"],
                        "class_name": det["class_name"],
                        "det_conf": det["det_conf"],
                        "is_hook": is_hook,
                        "result": result,
                    })

            tracks, next_track_id = update_temporal_tracks(
                tracks,
                display_items,
                next_track_id,
            )

            processed_frames += 1

        for track in tracks:
            draw_track(frame_bgr, track)

        if should_process or WRITE_SKIPPED_FRAMES:
            writer.write(frame_bgr)

        if frame_idx % 30 == 0:
            print(
                f"frame {frame_idx}/{total_frames} | "
                f"processed {processed_frames} | "
                f"hooks {total_hooks} | "
                f"unconnected {total_unconnected}"
            )

        frame_idx += 1

    cap.release()
    writer.release()

    print("=" * 70)
    print("Done")
    print("Output video:", OUTPUT_VIDEO)
    print("Processed frames:", processed_frames)
    print("Detected hooks:", total_hooks)
    print("Predicted unconnected hooks:", total_unconnected)
    if SAVE_DEBUG_CROPS:
        print("Debug crops:", DEBUG_CROP_DIR)

In [ ]:

# 12
run_video_pipeline()

In [ ]:
# 13
from google.colab import files
files.download(OUTPUT_VIDEO)